### Model Training

In [1]:
import numpy as np
import pandas as pd

In [22]:
df = pd.read_csv(r"C:\Users\shali\OneDrive\Desktop\FinanceX\data\FinanceX_classification_dataset_v2.csv")

In [23]:
df.head()

,customer_id,period,current_spending,days_elapsed,avg_daily_spending,income,previous_month_expense,days_remaining,projected_month_end_spending,monthly_budget,budget_usage_percentage,budget_remaining,budget_exceeded,category_spending
0,CUST000001,2023-01,4610.480733,12,384.206728,20241.309595,12734.370252,19,11910.408560,14866.559660,31.012425,10256.078927,0,1993.072505
1,CUST000001,2023-02,9195.580490,20,459.779025,21146.049414,12021.699821,8,12873.812686,13763.826457,66.809768,4568.245967,0,5747.000447
2,CUST000001,2023-03,6785.912445,17,399.171320,21094.469683,11508.859944,14,12374.310930,14581.446935,46.537991,7795.534489,0,3691.595907
3,CUST000001,2023-04,7746.950549,21,368.902407,20426.230029,13982.374219,9,11067.072213,13667.927056,56.679777,5920.976507,0,3182.577272
4,CUST000001,2023-05,4522.853885,12,376.904490,21124.380640,11559.300834,19,11684.039202,13874.879342,32.597429,9352.025458,0,1819.902670


In [25]:
df.drop(columns=["customer_id"], inplace=True)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360000 entries, 0 to 359999
Data columns (total 13 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   period                        360000 non-null  object 
 1   current_spending              360000 non-null  float64
 2   days_elapsed                  360000 non-null  int64  
 3   avg_daily_spending            360000 non-null  float64
 4   income                        360000 non-null  float64
 5   previous_month_expense        360000 non-null  float64
 6   days_remaining                360000 non-null  int64  
 7   projected_month_end_spending  360000 non-null  float64
 8   monthly_budget                360000 non-null  float64
 9   budget_usage_percentage       360000 non-null  float64
 10  budget_remaining              360000 non-null  float64
 11  budget_exceeded               360000 non-null  int64  
 12  category_spending             360000 non-nul

In [27]:
df['period'] = pd.to_datetime(df['period'], errors='coerce')
df['year'] = df['period'].dt.year
df['month'] = df['period'].dt.month

In [28]:
df = df.drop(columns=['period'])

In [29]:
x = df.drop(columns=["budget_exceeded"])
y = df["budget_exceeded"]

In [30]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

#### Pipeline

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [32]:
logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('logisticregression', LogisticRegression())
])

KNN_model = Pipeline([
    ('scaler', StandardScaler()),
    ('KNN', KNeighborsClassifier())
])

RandomForest_model = Pipeline([
    ('RandomForest', RandomForestClassifier(n_estimators=100, random_state=42))
])

Decision_model = Pipeline([
    ('DecisionTree', DecisionTreeClassifier(random_state=42))
])

GradientBoosting_model = Pipeline([
    ('GradientBoosting', GradientBoostingClassifier(n_estimators=100, random_state=42))
])

xgboost_model = Pipeline([
    ('xgboost', XGBClassifier(n_estimators=100, random_state=42))
])

models = {
    'Logistic Regression': logistic_model,
    'KNN': KNN_model,
    'Random Forest': RandomForest_model,
    'Decision Tree': Decision_model,
    'Gradient Boosting': GradientBoosting_model,
    'XGBoost': xgboost_model
}

In [33]:
results = []

for name, pipeline in models.items():
    pipeline.fit(x_train,y_train)

    y_train_pred = pipeline.predict(x_train)
    y_test_pred = pipeline.predict(x_test)

    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    report = classification_report(y_test, y_test_pred)
    matrix = confusion_matrix(y_test, y_test_pred)

    results.append({
        'Model': name,
        'Train Accuracy': train_accuracy,
        'Test Accuracy': test_accuracy,
        'Classification Report': report,
        'Confusion Matrix': matrix
    })

In [34]:
results

[{'Model': 'Logistic Regression',
  'Train Accuracy': 0.8410902777777778,
  'Test Accuracy': 0.8402083333333333,
  'Classification Report': '              precision    recall  f1-score   support\n\n           0       0.85      0.90      0.88     45851\n           1       0.81      0.73      0.77     26149\n\n    accuracy                           0.84     72000\n   macro avg       0.83      0.82      0.82     72000\nweighted avg       0.84      0.84      0.84     72000\n',
  'Confusion Matrix': array([[41421,  4430],
         [ 7075, 19074]])},
 {'Model': 'KNN',
  'Train Accuracy': 0.8702256944444444,
  'Test Accuracy': 0.8157361111111111,
  'Classification Report': '              precision    recall  f1-score   support\n\n           0       0.84      0.88      0.86     45851\n           1       0.77      0.70      0.74     26149\n\n    accuracy                           0.82     72000\n   macro avg       0.80      0.79      0.80     72000\nweighted avg       0.81      0.82      0.81  

In [46]:
best_model = models['XGBoost']
best_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('xgboost', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[<U28](13,)","['current_spending','days_elapsed','avg_daily_spending',..., 'category_spending','year','month']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None


In [ ]:
final_model = best_model
import joblib
joblib.dump(
    final_model,
    r"C:\Users\shali\OneDrive\Desktop\FinanceX\model\FinanceX_claassification_model.pkl"
)

['C:\\Users\\shali\\OneDrive\\Desktop\\FinanceX\\model\\FinanceX_claassification_model.pkl']